Ссылка для скачивания данных
https://cloud.mail.ru/public/mp8m/LL2xXnpxM

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
DATA = Path('../datasets')

import re
import requests
import pandas as pd
from io import BytesIO


def read_csv_from_mail_cloud(url, **read_csv_kwargs):
    """
    Загружает CSV-файл по публичной ссылке Mail Cloud
    и возвращает pandas.DataFrame.

    Parameters
    ----------
    url : str
        Публичная ссылка вида:
        https://cloud.mail.ru/public/...

    **read_csv_kwargs :
        Дополнительные параметры для pd.read_csv(),
        например sep=';', encoding='utf-8'.

    Returns
    -------
    pandas.DataFrame
    """

    # 1. Загружаем HTML публичной страницы
    page = requests.get(url)
    page.raise_for_status()

    # 2. Находим адрес download-сервера
    match = re.search(
        r'dispatcher.*?weblink_get.*?url"\s*:\s*"(.*?)"',
        page.text,
        flags=re.S
    )

    if match is None:
        raise RuntimeError(
            "Не удалось получить прямую ссылку на файл из Mail Cloud"
        )

    base_url = match.group(1)

    # 3. Берём идентификатор публичного файла
    public_id = url.split("/public/", 1)[1]

    download_url = f"{base_url}/{public_id}"

    # 4. Скачиваем сам файл
    response = requests.get(download_url)
    response.raise_for_status()

    # 5. Проверяем, что вместо файла не пришла HTML-страница
    content_type = response.headers.get("Content-Type", "")

    if "html" in content_type.lower():
        raise RuntimeError(
            "Mail Cloud вернул HTML-страницу вместо CSV-файла"
        )

    # 6. Читаем CSV
    return pd.read_csv(
        BytesIO(response.content),
        **read_csv_kwargs
    )

# СЕМИНАР 1 — Data Detective: от сырых данных к экономическому инсайту

**Множество решений до `model.fit()`, способных определить вывод**
  


1. Что является одной строкой / наблюдением?
2. Как сформирована выборка?
3. Что на самом деле означает переменная?
4. Как меняется вывод после агрегации, сегментации или преобразования?
5. Почему найденной закономерности пока нельзя доверять?

> **Prediction ≠ explanation ≠ causal inference.**


## 1. Доход домохозяйства и расходы на питание

**Вопрос:** как меняются расходы на питание по мере роста дохода домохозяйства?

In [2]:
engel = pd.read_csv(DATA / 'engel_food_expenditure_real.csv')
engel = read_csv_from_mail_cloud("https://cloud.mail.ru/public/Uiu7/vYGuSwtD9")

engel.head()

,income,foodexp
0,420.157651,255.839425
1,541.411707,310.958667
2,901.157457,485.680014
3,639.080229,402.997356
4,750.875606,495.560775


### 1.1 Разведочный анализ данных
Первый взгляд на данные

 - shape
 - dtypes 
 - missing values 
 - duplicated rows 
 - basic quantiles
 - duplicated
 - add data

In [ ]:
engel.duplicated().sum()

### 1.2 — data cleaning — не механическое удаление всего необычного.

Сравним среднее и медианное значение дохода.   
Построим график распределения доходов и график зависимости расходов на питание от дохода.


`Outlier` vs `Data error` vs `Influential observation`

Вопрос: является ли домохозяйство с самым высоким доходом очевидной ошибкой в данных или это может быть экономически осмысленное наблюдение? Стоит ли удалить это наблюдение? Какая доп информация может потребоватся?

### 1.3 Экономическая интерпретация

Закон Энгеля — эмпирическая закономерность в экономике потребления:
По мере роста дохода домохозяйства абсолютные расходы на питание обычно растут, но доля расходов на питание в доходе снижается.

Создадим переменную food_share = foodexp / income. 


---
## 2. Парадокс Симпсона. 

Cитуация, когда закономерность, наблюдаемая в агрегированных данных, существенно меняется или даже меняет направление после разделения данных на группы.

Перед нами агрегированные данные о поступлении в магистратуру University of California, Berkeley за 1973 год. 


- Admit: принят / не принят;
- Gender: мужчина / женщина;
- Dept: факультет/департамент A–F;
- Freq: количество заявителей в этой комбинации.


In [ ]:
berkeley = pd.read_csv(DATA / 'berkeley_admissions_1973_real.csv')
berkeley= read_csv_from_mail_cloud("https://cloud.mail.ru/public/BK4Z/RnKyzkmjP")

berkeley

### 2.1
Рассчитаем уровень поступления отдельно для мужчин и женщин. 

### 2.2
Рассчитаем долю принятых абитуриентов отдельно по факультетам и полу.


## Парадокс Симпсона. 

### Cитуация, когда закономерность, наблюдаемая в агрегированных данных, существенно меняется или даже меняет направление после разделения данных на группы.

---
## 3. Макроэкономические временные ряды

| Переменная | Смысл                                  |
| ---------- | -------------------------------------- |
| `date`     | конец квартала                         |
| `realgdp`  | реальный ВВП                           |
| `realcons` | реальное потребление                   |
| `realinv`  | реальные инвестиции                    |
| `realgovt` | реальные государственные расходы       |
| `realdpi`  | реальные располагаемые личные доходы   |
| `cpi`      | индекс потребительских цен, CPI        |
| `m1`       | денежный агрегат M1                    |
| `tbilrate` | ставка по краткосрочным Treasury Bills |
| `unemp`    | уровень безработицы, %                 |
| `pop`      | население                              |
| `infl`     | темп инфляции                          |
| `realint`  | реальная процентная ставка             |


In [ ]:
macro = pd.read_csv(DATA / 'us_macro_quarterly_real.csv', parse_dates=['date'])
macro = read_csv_from_mail_cloud("https://cloud.mail.ru/public/xyK7/Z9QJZr9gi")

macro.head()

### 3.1 Графики реального ВВП и реального потребления

Построим графики реального ВВП и реального потребления во времени. 

### 3.2 Преобразуем данные

Рассчитаем квартальные логарифмические темпы роста реального ВВП и потребления: $100\Delta\log X_t$. 
Сравним корреляции для уровней и для темпов роста.

Почему две временные серии с выраженным трендом могут выглядеть сильно связанными, даже если их краткосрочное совместное движение намного слабее?

## Какой коэффициент корреляции "правильный"?  0.9992 vs 0.6575?

---
## 4. Качество данных 


Банк хочет понять поведение клиентов и в будущем предсказывать, совершит ли клиент транзакцию в следующем месяце. Вам передали историю операций. Можно ли уже строить модель?

Перед вами синтетический датасет клиентских транзакций


| Поле                                   | Смысл                                                                                   |
| -------------------------------------- | --------------------------------------------------------------------------------------- |
| `client_id`                            | идентификатор клиента                                                                   |
| `date`                                 | дата транзакции                                                                         |
| `amount`                               | сумма транзакции                                                                        |
| `segment`                              | тип/сегмент клиента, например `regular`, `opportunistic`                                |
| `channel`                              | канал совершения операции, например `app`, `manager`                                    |
| `country`                              | страна клиента, например `RU`, `BY`, `AM`                                               |
| `declared_income`                      | заявленный доход клиента                                                                |
| `days_until_last_observed_transaction` | число дней от текущей транзакции до последней транзакции этого клиента во всём датасете |


In [ ]:
tx = pd.read_csv(DATA / 'client_transactions_pedagogical_synthetic.csv', parse_dates=['date'])
tx = read_csv_from_mail_cloud("https://cloud.mail.ru/public/2pQM/R12SB4KA5")
tx.head()

### 4.2 Пропущенный доход

Создадим индикатор пропущенных значений для declared_income. Сравните долю пропусков между различными segment и channel.

Что будет потеряно, если просто удалить все строки, в которых значение дохода отсутствует?

### 4.3  Единица наблюдения

Создадим признаки по клиентам: 
- количество транзакций, 
- сумму операций, 
- сумму положительных операций, 
- средний размер транзакции, 
- количество активных дней,
- дату первой и последней транзакции.

Как меняется исследовательский вопрос, когда единица наблюдения меняется с транзакции на клиента?

### 4.4 Поиск утечки данных

Изучите признак `days_until_last_observed_transaction`. Можно ли использовать его для прогнозирования того, совершит ли клиент ещё одну транзакцию в следующем месяце, если прогноз строится для исторических дат? 



## 4.5 Финальная задача
 
Вы получили этот датасет от бизнеса и должны через неделю показать первую модель прогнозирования активности клиента.

Какие пять вопросов вы зададите владельцу данных, на которые вы хотите получить ответ прежде, чем начнёте моделирование.